<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.

En este capítulo, predeciremos si la primera etapa del Falcon 9 aterrizará con éxito. SpaceX anuncia en su sitio web lanzamientos de cohetes Falcon 9 con un costo de 62 millones de dólares; Otros proveedores cuestan más de 165 millones de dólares cada uno; gran parte del ahorro se debe a que SpaceX puede reutilizar la primera etapa. Por lo tanto, si podemos determinar si la primera etapa aterrizará, podemos determinar el costo de un lanzamiento. Esta información se puede utilizar si una empresa alternativa quiere ofertar contra SpaceX para el lanzamiento de un cohete. En esta práctica de laboratorio, recopilará y se asegurará de que los datos estén en el formato correcto desde una API. El siguiente es un ejemplo de un lanzamiento exitoso.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:<br>
A continuación se muestran varios ejemplos de un aterrizaje fallido:

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. <br>

Se planean la mayoría de los aterrizajes fallidos. Space X realiza un aterrizaje controlado en los océanos.

## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data

<br>
En esta práctica de laboratorio, realizará una solicitud de obtención a la API de SpaceX. También realizarás algunas modificaciones y formateo de datos básicos.

- Solicitud a la API de SpaceX
- Limpiar los datos solicitados


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.<br>

A continuación definiremos una serie de funciones auxiliares que nos ayudarán a utilizar la API para extraer información utilizando números de identificación en los datos de lanzamiento.

De la columna de cohetes nos gustaría saber el nombre del propulsor.

In [19]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

#def getBoosterVersion(data):
#    # 1. Traer todos los cohetes en una sola petición
#    url_todos = "https://api.spacexdata.com/v4/rockets"
#    res = requests.get(url_todos)    
#    if res.status_code == 200:
#        # Crear un diccionario para mapear {id_cohete: nombre}
#        mapa_cohetes = {cohete['id']: cohete['name'] for cohete in res.json()}
#    else:
#        print(f"Error al conectar con la API de SpaceX: {res.status_code}")
#        mapa_cohetes = {}
#    # 2. Rrellenar la lista BoosterVersion buscando en el mapa local
#    for x in data['rocket']:
#        if x and x in mapa_cohetes:
#            BoosterVersion.append(mapa_cohetes[x])
#        else:
#            BoosterVersion.append(None) # Evita desalinear las filas del DataFrame

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.<br>
Desde la <code>launchpad</code> nos gustaría saber el nombre del sitio de lanzamiento que se está utilizando, la longitud y la latitud.


In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.<br>
De la <code>carga útil</code> nos gustaría conocer la masa de la carga útil y la órbita a la que se dirige.


In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.<br>
De los <code>núcleos</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese núcleo, si se usaron rejillas, si se reutilizó el núcleo, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del núcleo, que es un número que se usa para separar las versiones de los núcleos, el número de veces que este núcleo específico se ha reutilizado y el número de serie del núcleo.


In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:<br>
Ahora comencemos a solicitar datos de lanzamiento de cohetes desde la API de SpaceX con la siguiente URL:


In [6]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [7]:
response = requests.get(spacex_url)

Check the content of the response


In [8]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.<br>
Debería ver que la respuesta contiene información masiva sobre los lanzamientos de SpaceX. A continuación, intentemos descubrir información más relevante para este proyecto.


### Task 1: Request and parse the SpaceX launch data using the GET request
Tarea 1: Solicitar y analizar los datos del lanzamiento de SpaceX mediante la solicitud GET

To make the requested JSON results more consistent, we will use the following static response object for this project:<br>
Para que los resultados JSON solicitados sean más consistentes, usaremos el siguiente objeto de respuesta estática para este proyecto:


In [9]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code<br>
Deberíamos ver que la solicitud fue exitosa con el código de respuesta de estado 200.


In [10]:
response=requests.get(static_json_url)

In [11]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code><br>
Ahora decodificamos el contenido de la respuesta como Json usando <code>.json()</code> y lo convertimos en un marco de datos de Pandas usando <code>.json_normalize()</code>


In [12]:
# Use json_normalize meethod to convert the json result into a dataframe
df = pd.json_normalize(response.json())

Using the dataframe <code>data</code> print the first 5 rows


In [13]:
# Get the head of the dataframe
df.head(5)
data = df
#data.head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   static_fire_date_utc       100 non-null    object 
 1   static_fire_date_unix      100 non-null    float64
 2   tbd                        107 non-null    bool   
 3   net                        107 non-null    bool   
 4   window                     100 non-null    float64
 5   rocket                     107 non-null    object 
 6   success                    107 non-null    bool   
 7   details                    100 non-null    object 
 8   crew                       107 non-null    object 
 9   ships                      107 non-null    object 
 10  capsules                   107 non-null    object 
 11  payloads                   107 non-null    object 
 12  launchpad                  107 non-null    object 
 13  auto_update                107 non-null    bool   

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.<br>
Notarás que muchos de los datos son identificaciones. Por ejemplo, la columna del cohete no tiene información sobre el cohete, solo un número de identificación.

Ahora usaremos la API nuevamente para obtener información sobre los lanzamientos utilizando los ID proporcionados para cada lanzamiento. Específicamente usaremos las columnas <code>rocket</code>, <code>payloads</code>, <code>launchpad</code> y <code>cores</code>.



In [14]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe. <br>
* Del <code>cohete</code> nos gustaría saber el nombre del refuerzo.

* De la <code>payload</code> nos gustaría saber la masa de la carga útil y la órbita a la que se dirige.

* Desde la <code>launchpad</code> nos gustaría saber el nombre del sitio de lanzamiento que se está utilizando, la longitud y la latitud.

* **De <code>núcleos</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese núcleo, si se usaron rejillas, si se reutilizó el núcleo, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del núcleo, que es un número usado para separar las versiones de los núcleos, el número de veces que este núcleo específico se ha reutilizado y el número de serie del núcleo.**

Los datos de estas solicitudes se almacenarán en listas y se utilizarán para crear un nuevo marco de datos.


In [21]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:<br>
Estas funciones aplicarán las salidas globalmente a las variables anteriores. Echemos un vistazo a la variable <code>BoosterVersion</code>. Antes de aplicar <code>getBoosterVersion</code> la lista está vacía:


In [22]:
BoosterVersion

[]

brNow, let's apply <code> getBoosterVersion</code> function method to get the booster version<br>
Ahora, apliquemos el método de función <code> getBoosterVersion</code> para obtener la versión de refuerzo.


In [23]:
# Call getBoosterVersion
# esta dando error 525, vamos a probar mas tarde
getBoosterVersion(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

the list has now been update 


In [24]:
BoosterVersion[0:5]

[]

we can apply the rest of the  functions here:


In [25]:
# Call getLaunchSite
getLaunchSite(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [26]:
# Call getPayloadData
getPayloadData(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [27]:
# Call getCoreData
getCoreData(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.<br>
Finalmente, construyamos nuestro conjunto de datos utilizando los datos que hemos obtenido. Combinamos las columnas en un diccionario.


In [28]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.<br>
Luego, necesitamos crear un marco de datos de Pandas a partir del diccionario launch_dict.


In [29]:
# Comprobar la longitud de cada lista en el diccionario # esta es una celda nueva
for clave, lista in launch_dict.items():
    print(f"{clave}: {len(lista)} elementos")

FlightNumber: 94 elementos
Date: 94 elementos
BoosterVersion: 0 elementos
PayloadMass: 0 elementos
Orbit: 0 elementos
LaunchSite: 0 elementos
Outcome: 0 elementos
Flights: 0 elementos
GridFins: 0 elementos
Reused: 0 elementos
Legs: 0 elementos
LandingPad: 0 elementos
Block: 0 elementos
ReusedCount: 0 elementos
Serial: 0 elementos
Longitude: 0 elementos
Latitude: 0 elementos


In [30]:
# Create a data from launch_dict
df_ld = pd.DataFrame(launch_dict)

ValueError: All arrays must be of the same length

Show the summary of the dataframe<br>
Mostrar el resumen del dataframe


In [31]:
# Show the head of the dataframe
df_ld.head(5)

NameError: name 'df_ld' is not defined

### Task 2: Filter the dataframe to only include `Falcon 9` launches
Tarea 2: filtrar el marco de datos para incluir solo los lanzamientos de "Falcon 9"


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.<br>
Finalmente eliminaremos los lanzamientos de Falcon 1 manteniendo solo los lanzamientos de Falcon 9. Filtre el marco de datos usando la columna <code>BoosterVersion</code> para conservar solo los lanzamientos de Falcon 9. Guarde los datos filtrados en un nuevo marco de datos llamado <code>data_falcon9</code>.

In [2]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9=df_ld[df_ld['BoosterVersion']!='Falcon 1']

NameError: name 'df_ld' is not defined

Now that we have removed some values we should reset the FlgihtNumber column<br>
Ahora que hemos eliminado algunos valores, debemos restablecer la columna FlgihtNumber.


In [3]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

NameError: name 'data_falcon9' is not defined

## Data Wrangling
Disputa de datos

We can see below that some of the rows are missing values in our dataset.<br>
Podemos ver a continuación que a algunas de las filas les faltan valores en nuestro conjunto de datos.


In [ ]:
data_falcon9.isnull().sum()

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.<br>
Antes de que podamos continuar, debemos abordar estos valores faltantes. La columna <code>LandingPad</code> conservará los valores Ninguno para representar cuándo no se utilizaron las plataformas de aterrizaje.


### Task 3: Dealing with Missing Values - Lidiar con los valores perdidos


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.<br>
Calcule a continuación la media para <code>PayloadMass</code> usando <code>.mean()</code>. Luego use la media y la función <code>.replace()</code> para reemplazar los valores `np.nan` en los datos con la media que calculó.


In [ ]:
# Calculate the mean value of PayloadMass column
# Replace the np.nan values with its mean value
x=data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'].replace(np.nan,x, inplace=True)
data_falcon9.isnull().sum()



You should see the number of missing values of the <code>PayLoadMass</code> change to zero.<br>
Debería ver que el número de valores faltantes de <code>PayLoadMass</code> cambia a cero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.<br>
Ahora no deberíamos tener valores faltantes en nuestro conjunto de datos excepto en <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range.<br> 
Ahora podemos exportarlo a un <b>CSV</b> para la siguiente sección, pero para que las respuestas sean consistentes, en la próxima práctica de laboratorio proporcionaremos datos en un rango de fechas preseleccionado.


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
